# Feedback Round 2 Notebook

This notebook contains the end-to-end pipeline for the Round 2 feedback tasks:
1. canonical NFCorpus recomputation,
2. BioASQ subset construction,
3. BGE cross-encoder reranking,
4. per-query router,
5. MIRAGE downstream evaluation,
6. bounds/gap support tables.


In [ ]:
# Single-stage install. With pylate gone we no longer need the
# multi-stage / force-reinstall / kernel-suicide gymnastics that earlier
# revisions of this cell carried.
!pip install -q \
    "beir==2.0.0" "bm25s==0.2.6" "datasets>=2.18.0" \
    "scikit-learn>=1.3.0" "lightgbm>=4.0.0" "scipy>=1.11.0" \
    "faiss-cpu>=1.7.4" "rank-bm25>=0.2.2" "rouge-score>=0.1.2" \
    "nltk>=3.8.0" "numpy>=1.24.0" "pandas>=2.0.0" "psutil>=5.9.0" \
    "tqdm>=4.65.0" "sentencepiece" \
    "transformers>=4.45.0,<5" "sentence-transformers>=3.0.0,<5" \
    "bitsandbytes>=0.43.0" "accelerate>=0.34.0"
print("\nDependencies installed. Continue with cell 2.")

In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path('/Users/dmitrijnaumov/Code/Inno/AdvancedML/RAGEval')
OUT = ROOT / 'notebooks' / 'results' / 'feedback2'
for sub in ['tables', 'datasets', 'figures', 'cache']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print('output root:', OUT)


output root: /Users/dmitrijnaumov/Code/Inno/AdvancedML/RAGEval/notebooks/results/feedback2


In [2]:
# Canonical NFCorpus table export (sourced from current validated run artifacts)
hybrid = pd.read_csv(ROOT / 'notebooks' / 'results' / 'next_stage' / 'tables' / 'hybrid_test_comparison.csv')
canon = pd.DataFrame([
    {'Method': 'BM25', 'nDCG@10': 0.2939534468857542},
    {'Method': 'Dense', 'nDCG@10': 0.3172660881458678},
    {'Method': 'BGE-small', 'nDCG@10': 0.3388},
    {'Method': 'E5-small', 'nDCG@10': 0.3267},
    {'Method': 'SPLADE', 'nDCG@10': 0.3327},
    {'Method': 'MedCPT', 'nDCG@10': 0.3246},
])
canon.to_csv(OUT / 'tables' / 'nfcorpus_canonical.csv', index=False)
canon


,Method,nDCG@10
0,BM25,0.293953
1,Dense,0.317266
2,BGE-small,0.338800
3,E5-small,0.326700
4,SPLADE,0.332700
5,MedCPT,0.324600


In [3]:
# n-required summary table for pairwise distinguishability under Hoeffding
delta_rows = [
    ('BGE-small', 'BM25', 0.3388 - 0.2940),
    ('BGE-small', 'E5-small', 0.3388 - 0.3267),
    ('Hybrid', 'Dense', 0.33804189953671526 - 0.3172660881458678),
    ('Dense', 'BM25', 0.3172660881458678 - 0.2939534468857542),
]
delta = 0.05
records = []
for a, b, d in delta_rows:
    if abs(d) < 1e-12:
        n_req = math.inf
    else:
        n_req = math.ceil(math.log(2 / delta) / (2 * (d ** 2)))
    records.append({'A': a, 'B': b, 'observed_gap': d, 'hoeffding_n_required_95pct': n_req})
pd.DataFrame(records).to_csv(OUT / 'tables' / 'n_required.csv', index=False)
pd.DataFrame(records)


,A,B,observed_gap,hoeffding_n_required_95pct
0,BGE-small,BM25,0.044800,919
1,BGE-small,E5-small,0.012100,12598
2,Hybrid,Dense,0.020776,4274
3,Dense,BM25,0.023313,3394


---

## Phase-B common helpers (Colab-friendly)

The cells below are intended to be executed on Google Colab T4. They install
all dependencies, define a single set of retrieval helpers, and lazily load
the four base models (BM25 / MiniLM / BGE / E5 / SPLADE / MedCPT). Each
Phase-B section reuses these helpers.

If the notebook is run from the local repo (no `/content` directory),
`ROOT` falls back to the local path that was set in cell 1.


In [ ]:
# Colab-friendly setup -----------------------------------------------------
import os, sys, subprocess, importlib

IN_COLAB = os.path.exists("/content")

# Ensure all needed packages are installed (idempotent)
def _pip(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

_required = [
    "beir==2.0.0",
    "bm25s==0.2.6",
    "sentence-transformers>=2.7.0",
    "transformers>=4.43.0",
    "scikit-learn>=1.3.0",
    "lightgbm>=4.0.0",
    "scipy>=1.11.0",
    "datasets>=2.18.0",
    "psutil",
    "tqdm",
]
try:
    import beir, bm25s, sentence_transformers, transformers, sklearn, lightgbm, scipy, datasets  # noqa: F401
except ImportError:
    print("Installing core dependencies ...")
    _pip(_required)

import gc, time, json, pickle, math, random
import numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
import torch, torch.nn.functional as F
from tqdm.auto import tqdm

import transformers
print(f"transformers {transformers.__version__} -- OK")

# beir.util.download_and_unzip is used by both B.1 (NFCorpus / TREC-COVID /
# SciFact / ArguAna BEIR downloads) and B.3 (NFCorpus reload for the router).
# Importing once here so every downstream cell can reach `beir_util.*`.
from beir import util as beir_util  # noqa: F401
from beir.datasets.data_loader import GenericDataLoader  # noqa: F401

# If we are on Colab and ROOT is unset, mount Drive and pin to a project folder.
if IN_COLAB and ("ROOT" not in globals() or not Path(ROOT).exists()):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/RAGEval")
    ROOT.mkdir(parents=True, exist_ok=True)
    OUT = ROOT / "notebooks" / "results" / "feedback2"
    for sub in ["tables", "datasets", "figures", "cache"]:
        (OUT / sub).mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| ROOT:", ROOT)

DATASETS_DIR = ROOT / "notebooks" / "datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


# Metric helpers (mirrors final.ipynb / cell 13) --------------------------
def compute_ndcg(ranked_docs, qrel, k):
    dcg = 0.0
    for i, did in enumerate(ranked_docs[:k]):
        rel = qrel.get(did, 0)
        dcg += (2 ** rel - 1) / np.log2(i + 2)
    ideal = sorted(qrel.values(), reverse=True)[:k]
    idcg = sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def compute_recall(ranked_docs, qrel, k, threshold=1):
    relevant = {d for d, r in qrel.items() if r >= threshold}
    if not relevant:
        return 0.0
    found = sum(1 for d in ranked_docs[:k] if d in relevant)
    return found / len(relevant)


def compute_precision(ranked_docs, qrel, k, threshold=1):
    relevant = {d for d, r in qrel.items() if r >= threshold}
    if k == 0:
        return 0.0
    found = sum(1 for d in ranked_docs[:k] if d in relevant)
    return found / k


def evaluate_retriever(results, qrels, k_values=(1, 3, 5, 10)):
    per_query = {}
    for qid in qrels:
        if qid not in results:
            continue
        ranking = [d for d, _ in sorted(results[qid].items(), key=lambda x: -x[1])]
        m = {}
        for k in k_values:
            m[f"nDCG@{k}"] = compute_ndcg(ranking, qrels[qid], k)
            m[f"Recall@{k}"] = compute_recall(ranking, qrels[qid], k)
            m[f"P@{k}"] = compute_precision(ranking, qrels[qid], k)
        per_query[qid] = m
    if not per_query:
        return {"aggregate": {}, "per_query": {}, "loss_array": np.array([])}
    keys = list(next(iter(per_query.values())).keys())
    agg = {k: float(np.mean([m[k] for m in per_query.values()])) for k in keys}
    loss = np.array([1.0 - m["nDCG@10"] for m in per_query.values()])
    return {"aggregate": agg, "per_query": per_query, "loss_array": loss}


# Retrieval helpers (BM25 / dense / SPLADE / MedCPT) ----------------------
def _prep_corpus(corpus):
    ids = list(corpus.keys())
    texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in ids]
    return ids, texts


def _bm25_run(doc_ids, doc_texts, queries, top_k=100):
    """BM25 over `queries` with bm25s.

    NOTE: `bm.retrieve(...)` is forced to single-threaded execution
    (`n_threads=1`). bm25s 0.2.x defaults to a multiprocessing pool and on
    Colab kernels this can return zero items from the worker pool, which
    surfaces as `ValueError: not enough values to unpack (expected 2, got 0)`
    inside bm25s. Serial execution is fast enough for our corpus sizes
    (<= 50K docs) and is reproducible across environments.
    """
    import bm25s
    if not queries:
        return {}
    corpus_tok = bm25s.tokenize(doc_texts, stopwords="en", show_progress=False)
    bm = bm25s.BM25(k1=1.5, b=0.75)
    bm.index(corpus_tok)
    qids = list(queries.keys())
    qtok = bm25s.tokenize([queries[q] for q in qids], stopwords="en",
                          show_progress=False)
    k = min(top_k, len(doc_ids))
    try:
        res, sc = bm.retrieve(qtok, k=k, n_threads=1, show_progress=False)
    except TypeError:
        # Older bm25s without n_threads / show_progress kwargs.
        res, sc = bm.retrieve(qtok, k=k)
    out = {}
    for i, qid in enumerate(qids):
        out[qid] = {doc_ids[res[i][j]]: float(sc[i][j]) for j in range(k)}
    return out


def _dense_run(doc_ids, doc_texts, queries, model, qpfx="", dpfx="", top_k=100, batch=64):
    if not doc_ids or not queries:
        return {qid: {} for qid in queries}
    corpus_texts = [dpfx + t for t in doc_texts] if dpfx else doc_texts
    doc_emb = model.encode(corpus_texts, batch_size=batch, show_progress_bar=False,
                           normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    qids = list(queries.keys())
    qtxt = [(qpfx + queries[q]) if qpfx else queries[q] for q in qids]
    q_emb = model.encode(qtxt, batch_size=128, normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    if doc_emb.ndim != 2 or q_emb.ndim != 2:
        raise RuntimeError(
            f"Dense encoder produced non-2D embeddings (doc={doc_emb.shape}, "
            f"q={q_emb.shape}); check that doc_texts and queries are non-empty "
            "lists of non-empty strings.")
    scores = q_emb @ doc_emb.T
    k = min(top_k, len(doc_ids))
    out = {}
    for i, qid in enumerate(qids):
        top = np.argpartition(scores[i], -k)[-k:]
        top = top[np.argsort(-scores[i][top])]
        out[qid] = {doc_ids[j]: float(scores[i][j]) for j in top}
    del doc_emb, q_emb, scores; gc.collect()
    return out


def _splade_run(doc_ids, doc_texts, queries, tok, mod, top_k=100, bs=8, max_len=128):
    if not doc_ids or not queries:
        return {qid: {} for qid in queries}
    def _enc(texts):
        rows = []
        for i in range(0, len(texts), bs):
            inp = tok(texts[i:i + bs], return_tensors="pt", padding=True,
                      truncation=True, max_length=max_len)
            inp = {k: v.to(device) for k, v in inp.items()}
            with torch.no_grad():
                vecs = torch.log(1 + torch.relu(mod(**inp).logits))
            vecs = (vecs * inp["attention_mask"].unsqueeze(-1)).max(dim=1).values
            rows.append(sp.csr_matrix(vecs.cpu().numpy()))
        return sp.vstack(rows)
    doc_vecs = _enc(doc_texts)
    qids = list(queries.keys())
    q_vecs = _enc([queries[q] for q in qids])
    scores = (q_vecs @ doc_vecs.T).toarray()
    n_docs = len(doc_ids)
    k = min(top_k, n_docs)
    out = {}
    for i, qid in enumerate(qids):
        # np.argpartition requires kth in [0, n-1]. When k == n we have to
        # take the full sorted ranking instead.
        if k >= n_docs:
            top = np.argsort(-scores[i])
        else:
            top = np.argpartition(-scores[i], k)[:k]
            top = top[np.argsort(-scores[i][top])]
        out[qid] = {doc_ids[j]: float(scores[i][j]) for j in top}
    del doc_vecs, q_vecs, scores; gc.collect()
    return out


def _medcpt_run(doc_ids, doc_texts, queries, qry_tok, qry_mod, art_tok, art_mod, top_k=100, bs=16):
    if not doc_ids or not queries:
        return {qid: {} for qid in queries}
    def _enc(texts, tok, mod, max_len):
        embs = []
        for i in range(0, len(texts), bs):
            inp = tok(texts[i:i + bs], return_tensors="pt", padding=True,
                      truncation=True, max_length=max_len)
            inp = {k: v.to(device) for k, v in inp.items()}
            with torch.no_grad():
                e = F.normalize(mod(**inp).last_hidden_state[:, 0, :], dim=-1)
            embs.append(e.cpu().numpy())
        return np.concatenate(embs, axis=0).astype("float32")
    doc_emb = _enc(doc_texts, art_tok, art_mod, 256)
    qids = list(queries.keys())
    q_emb = _enc([queries[q] for q in qids], qry_tok, qry_mod, 64)
    scores = q_emb @ doc_emb.T
    k = min(top_k, len(doc_ids))
    out = {}
    for i, qid in enumerate(qids):
        top = np.argpartition(scores[i], -k)[-k:]
        top = top[np.argsort(-scores[i][top])]
        out[qid] = {doc_ids[j]: float(scores[i][j]) for j in top}
    del doc_emb, q_emb, scores; gc.collect()
    return out


# Lazy model loaders ------------------------------------------------------
_MODELS = {}


def get_minilm():
    if "minilm" not in _MODELS:
        from sentence_transformers import SentenceTransformer
        _MODELS["minilm"] = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
    return _MODELS["minilm"]


def get_bge():
    if "bge" not in _MODELS:
        from sentence_transformers import SentenceTransformer
        _MODELS["bge"] = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
    return _MODELS["bge"]


def get_e5():
    if "e5" not in _MODELS:
        from sentence_transformers import SentenceTransformer
        _MODELS["e5"] = SentenceTransformer("intfloat/e5-small-v2", device=device)
    return _MODELS["e5"]


def get_splade():
    if "splade" not in _MODELS:
        from transformers import AutoTokenizer, AutoModelForMaskedLM
        name = "naver/splade-cocondenser-ensembledistil"
        tok = AutoTokenizer.from_pretrained(name)
        mod = AutoModelForMaskedLM.from_pretrained(name).to(device).eval()
        _MODELS["splade"] = (tok, mod)
    return _MODELS["splade"]


def get_medcpt():
    if "medcpt" not in _MODELS:
        from transformers import AutoTokenizer, AutoModel
        qt = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
        qm = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder").to(device).eval()
        at = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
        am = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(device).eval()
        _MODELS["medcpt"] = (qt, qm, at, am)
    return _MODELS["medcpt"]


print("Phase-B helpers ready.")


---

## B.1 BioASQ-subset retrieval

Builds the deterministic 50K-document BioASQ-BEIR subset (qrel-union plus
distractors), samples 500 test queries, and evaluates all six retrievers.
Outputs:

- `notebooks/results/feedback2/datasets/bioasq_subset_doc_ids.json`
  (replaces the existing protocol-only stub with materialised IDs)
- `notebooks/results/feedback2/cache/bioasq_perquery.parquet`
  (per-query nDCG@10 for every retriever)
- `notebooks/results/feedback2/tables/multi_dataset_ndcg10_v2.csv`
  (BioASQ row backfilled)
- `notebooks/results/feedback2/tables/bioasq_paired_bootstrap.csv`
  (paired-bootstrap CI for MedCPT minus BM25 — the central biomedical claim)


In [ ]:
# B.1.a Load BioASQ and build a deterministic ~50K-doc subset.
#
# IMPORTANT: BEIR's UKP-DARMSTADT mirror does NOT host the BioASQ corpus
# (license reasons; the .zip there is an HTML placeholder, which triggers a
# `BadZipFile` error). We therefore use a multi-source loader with the
# following preference order:
#
#   1. A locally-prepared BEIR-format directory at DATASETS_DIR/bioasq/
#      (corpus.jsonl, queries.jsonl, qrels/test.tsv) -- if the user has
#      already downloaded BioASQ from the official challenge site.
#   2. HuggingFace `BeIR/bioasq` + `BeIR/bioasq-qrels` (BEIR-format
#      community mirror; works without registration).
#   3. HuggingFace `rag-datasets/rag-mini-bioasq` (small ~4K-doc public
#      mirror; always accessible, used as a last resort so the rest of the
#      pipeline still runs end-to-end).
#
# In every case we end up with `_corpus_full`, `_queries_full`,
# `_qrels_full` in BEIR's dict-of-dicts format, so the subset construction
# below is identical regardless of source.
# `GenericDataLoader` is imported in the setup cell above.

BIOASQ_DIR = DATASETS_DIR / "bioasq"
SEED = 42
TARGET_SIZE = 50_000


def _load_bioasq_local():
    if not (BIOASQ_DIR / "corpus.jsonl").exists():
        return None
    print(f"  using local BEIR-format BioASQ at {BIOASQ_DIR}")
    return GenericDataLoader(str(BIOASQ_DIR)).load(split="test")


def _load_bioasq_hf_beir():
    """Load BEIR-format BioASQ from HuggingFace mirror `BeIR/bioasq`."""
    from datasets import load_dataset
    print("  trying HuggingFace BeIR/bioasq mirror ...", flush=True)
    corpus_ds = load_dataset("BeIR/bioasq", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/bioasq", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/bioasq-qrels", split="test")

    corpus = {}
    for r in corpus_ds:
        corpus[str(r["_id"])] = {
            "title": r.get("title", "") or "",
            "text": r.get("text", "") or "",
        }
    queries = {str(r["_id"]): r["text"] for r in queries_ds}
    qrels = {}
    for r in qrels_ds:
        qid, did, score = str(r["query-id"]), str(r["corpus-id"]), int(r["score"])
        qrels.setdefault(qid, {})[did] = score
    return corpus, queries, qrels


def _load_bioasq_mini():
    """Fallback: small public BioASQ mirror. Tries `enelpol/rag-mini-bioasq`
    first (cleaned: dedup'd passages, no NaNs, `relevant_passage_ids` is a
    proper sequence-of-int) and falls back to `rag-datasets/rag-mini-bioasq`
    (original mirror, where `relevant_passage_ids` is a JSON string).
    Yields ~40K passages and ~3K-4K QAP triples."""
    from datasets import load_dataset
    import ast as _ast

    last_exc = None
    for repo in ("enelpol/rag-mini-bioasq", "rag-datasets/rag-mini-bioasq"):
        try:
            print(f"  falling back to {repo} ...", flush=True)
            corpus_ds = load_dataset(repo, "text-corpus", split="passages")
            try:
                qa_ds = load_dataset(repo, "question-answer-passages",
                                     split="test")
            except Exception:
                qa_ds = load_dataset(repo, "question-answer-passages",
                                     split="train")
        except Exception as exc:
            last_exc = exc
            continue

        corpus = {}
        for r in corpus_ds:
            did = str(r.get("id"))
            text = r.get("passage", "") or ""
            if not text or text == "nan":
                continue
            corpus[did] = {"title": "", "text": text}
        if not corpus:
            continue

        queries, qrels = {}, {}
        for i, r in enumerate(qa_ds):
            qid = str(r.get("id", i))
            q = r.get("question", "") or ""
            rel = r.get("relevant_passage_ids", []) or []
            if isinstance(rel, str):
                try:
                    rel = json.loads(rel)
                except (ValueError, TypeError):
                    try:
                        rel = _ast.literal_eval(rel)
                    except (ValueError, SyntaxError):
                        rel = []
            rel_ids = [str(d) for d in rel if str(d) in corpus]
            if not rel_ids or not q:
                continue
            queries[qid] = q
            qrels[qid] = {d: 1 for d in rel_ids}

        if not queries:
            continue
        return corpus, queries, qrels

    if last_exc is not None:
        raise last_exc
    raise RuntimeError("rag-mini-bioasq mirrors loaded but produced no QAP triples")


_corpus_full = _queries_full = _qrels_full = None
BIOASQ_SOURCE = None
_loaders = [
    ("local BEIR directory",      _load_bioasq_local),
    ("HuggingFace BeIR/bioasq",   _load_bioasq_hf_beir),
    ("rag-mini-bioasq fallback",  _load_bioasq_mini),
]
for _label, _fn in _loaders:
    try:
        _result = _fn()
    except Exception as _exc:
        print(f"  [{_label}] failed: {type(_exc).__name__}: {_exc}")
        continue
    if _result is None:
        continue
    _corpus_full, _queries_full, _qrels_full = _result
    BIOASQ_SOURCE = _label
    break

if _corpus_full is None:
    raise RuntimeError(
        "All BioASQ loaders failed. To use the official corpus, register at "
        "http://bioasq.org/, download the BEIR-format release, and unpack "
        f"it into {BIOASQ_DIR} so it contains corpus.jsonl, queries.jsonl, "
        "and qrels/test.tsv."
    )

print(f"  source: {BIOASQ_SOURCE}")
print(f"  full corpus: {len(_corpus_full):,} docs  |  "
      f"{len(_queries_full):,} queries  |  qrels: {len(_qrels_full):,}")

# Adjust subset target if we are on the small fallback mirror.
TARGET_SIZE = min(TARGET_SIZE, len(_corpus_full))

rng = np.random.default_rng(SEED)
qrel_union = set()
for qid, rels in _qrels_full.items():
    qrel_union.update(rels.keys())
qrel_union &= set(_corpus_full.keys())
print(f"  qrel-union size: {len(qrel_union):,}")

remaining = list(set(_corpus_full.keys()) - qrel_union)
n_distractors = max(0, min(TARGET_SIZE - len(qrel_union), 10_000, len(remaining)))
distractor_ids = (rng.choice(remaining, size=n_distractors, replace=False).tolist()
                  if n_distractors > 0 else [])

subset_ids = sorted(qrel_union | set(distractor_ids))
print(f"  subset size: {len(subset_ids):,} "
      f"({len(qrel_union):,} qrel-union + {n_distractors:,} distractors)")

manifest = {
    "seed": SEED,
    "source": BIOASQ_SOURCE,
    "protocol": "union_of_qrel_relevant_docs_plus_distractors",
    "target_size": TARGET_SIZE,
    "n_qrel_union": len(qrel_union),
    "n_distractors": n_distractors,
    "n_total": len(subset_ids),
    "doc_ids": subset_ids,
}
manifest_path = OUT / "datasets" / "bioasq_subset_doc_ids.json"
with open(manifest_path, "w") as fh:
    json.dump(manifest, fh)
print(f"  manifest saved -> {manifest_path}")

# Hard sanity check: silent empties downstream in `_dense_run` produce a
# confusing `(N, 0)` matmul error, so we fail here instead.
assert len(subset_ids) > 0, (
    f"BioASQ subset is empty (source={BIOASQ_SOURCE!r}). "
    "Check the loader: corpus / qrels parsing likely produced 0 docs.")
assert len(_qrels_full) > 0, (
    f"No qrels parsed from source={BIOASQ_SOURCE!r}; downstream evaluation "
    "would be vacuous. Inspect _load_bioasq_* outputs.")


In [ ]:
# B.1.b Build the subset corpus and a 500-query deterministic sample.
subset_set = set(subset_ids)
bioasq_corpus = {d: _corpus_full[d] for d in subset_ids}

# Drop queries with zero relevant docs in the subset.
bioasq_qrels = {}
for qid, rels in _qrels_full.items():
    inner = {d: r for d, r in rels.items() if d in subset_set}
    if inner:
        bioasq_qrels[qid] = inner

eligible_qids = sorted(bioasq_qrels.keys())
rng = np.random.default_rng(SEED)
sample_size = min(500, len(eligible_qids))
sampled = sorted(rng.choice(eligible_qids, size=sample_size, replace=False).tolist())
bioasq_queries = {qid: _queries_full[qid] for qid in sampled if qid in _queries_full}
bioasq_qrels = {qid: bioasq_qrels[qid] for qid in bioasq_queries}
print(f"  sampled {len(bioasq_queries)} queries | qrel-coverage in subset OK")

doc_ids_b, doc_texts_b = _prep_corpus(bioasq_corpus)
print(f"  prepared {len(doc_ids_b):,} documents for retrieval")

assert len(bioasq_queries) > 0, (
    "BioASQ query sample is empty -- every query had zero qrels in the "
    "subset. This usually means the loader couldn't parse "
    "`relevant_passage_ids` (rag-mini-bioasq stores it as a JSON string). "
    "Re-run cell B.1.a after restarting the kernel.")
assert len(doc_ids_b) > 0, (
    "BioASQ subset corpus is empty. Re-run cell B.1.a; the loader probably "
    "fell through to an empty mirror.")


In [ ]:
# B.1.c Run all 6 retrievers on the BioASQ subset (with disk caching).
BIOASQ_CACHE = CACHE_DIR / "bioasq_results.pkl"
EXPECTED_RETRIEVERS = ["BM25", "Dense", "BGE-small", "E5-small", "SPLADE", "MedCPT"]


def _cache_complete(blob, query_ids):
    """A cache is usable only if every retriever ran on every query and
    produced at least one ranked doc per query (empty {qid: {}} dicts are
    the silent-failure mode we want to retry, not reuse)."""
    if not isinstance(blob, dict):
        return False
    if set(blob.keys()) != set(EXPECTED_RETRIEVERS):
        return False
    qid_set = set(query_ids)
    for name, res in blob.items():
        if not isinstance(res, dict) or set(res.keys()) != qid_set:
            return False
        if not any(res[qid] for qid in qid_set):
            return False
    return True


bioasq_results = None
if BIOASQ_CACHE.exists():
    try:
        with open(BIOASQ_CACHE, "rb") as fh:
            _cached = pickle.load(fh)
        if _cache_complete(_cached, bioasq_queries.keys()):
            bioasq_results = _cached
            print("Loaded BioASQ retrieval results from cache.")
        else:
            print("BioASQ cache is incomplete or empty; re-running retrievers.")
    except Exception as exc:
        print(f"BioASQ cache unreadable ({exc}); re-running retrievers.")

if bioasq_results is None:
    bioasq_results = {}

    print("BM25 ...", flush=True)
    bioasq_results["BM25"] = _bm25_run(doc_ids_b, doc_texts_b, bioasq_queries, top_k=100)

    print("Dense (MiniLM) ...", flush=True)
    bioasq_results["Dense"] = _dense_run(doc_ids_b, doc_texts_b, bioasq_queries,
                                         get_minilm(), top_k=100)
    print("BGE-small ...", flush=True)
    bioasq_results["BGE-small"] = _dense_run(doc_ids_b, doc_texts_b, bioasq_queries,
                                              get_bge(), top_k=100)
    print("E5-small ...", flush=True)
    bioasq_results["E5-small"] = _dense_run(doc_ids_b, doc_texts_b, bioasq_queries,
                                             get_e5(),
                                             qpfx="query: ", dpfx="passage: ", top_k=100)
    print("SPLADE ...", flush=True)
    _splade_tok, _splade_mod = get_splade()
    bioasq_results["SPLADE"] = _splade_run(doc_ids_b, doc_texts_b, bioasq_queries,
                                            _splade_tok, _splade_mod, top_k=100)
    print("MedCPT ...", flush=True)
    _qt, _qm, _at, _am = get_medcpt()
    bioasq_results["MedCPT"] = _medcpt_run(doc_ids_b, doc_texts_b, bioasq_queries,
                                            _qt, _qm, _at, _am, top_k=100)
    with open(BIOASQ_CACHE, "wb") as fh:
        pickle.dump(bioasq_results, fh)

bioasq_eval = {name: evaluate_retriever(res, bioasq_qrels) for name, res in bioasq_results.items()}
for name, ev in bioasq_eval.items():
    print(f"  {name:<10} nDCG@10 = {ev['aggregate']['nDCG@10']:.4f}")


In [ ]:
# B.1.d Backfill multi_dataset_ndcg10_v2.csv and emit paired bootstrap CI.
md_path = OUT / "tables" / "multi_dataset_ndcg10_v2.csv"
md = pd.read_csv(md_path)
mask = md["Dataset"] == "BioASQ-subset"
for col, name in [
    ("BM25 nDCG@10", "BM25"),
    ("Dense nDCG@10", "Dense"),
    ("BGE-small nDCG@10", "BGE-small"),
    ("E5-small nDCG@10", "E5-small"),
    ("SPLADE nDCG@10", "SPLADE"),
    ("MedCPT nDCG@10", "MedCPT"),
]:
    md.loc[mask, col] = round(bioasq_eval[name]["aggregate"]["nDCG@10"], 4)
md.loc[mask, "# Docs"] = len(doc_ids_b)
md.loc[mask, "# Queries"] = len(bioasq_queries)
md.loc[mask, "Notes"] = (f"Subset of qrel-union ({len(qrel_union):,}) + "
                        f"{n_distractors:,} distractors, seed=42, 500-query sample")
md.to_csv(md_path, index=False)
print(f"Updated {md_path}")
print(md)


# Per-query nDCG@10 -> parquet (used later by the router)
qids = sorted(bioasq_queries.keys())
perquery = {"qid": qids}
for name, ev in bioasq_eval.items():
    perquery[name] = [ev["per_query"][q]["nDCG@10"] for q in qids]
pq_df = pd.DataFrame(perquery)
pq_df.to_parquet(CACHE_DIR / "bioasq_perquery.parquet", index=False)
print(f"Per-query nDCG@10 -> {CACHE_DIR / 'bioasq_perquery.parquet'}")


# Paired bootstrap CI: MedCPT - BM25 on BioASQ
def paired_bootstrap_ci(a, b, B=10_000, alpha=0.05, seed=42):
    a, b = np.asarray(a, float), np.asarray(b, float)
    diffs = a - b
    rng = np.random.default_rng(seed)
    n = len(diffs)
    boot = np.array([diffs[rng.integers(0, n, n)].mean() for _ in range(B)])
    lo, hi = np.quantile(boot, [alpha / 2, 1 - alpha / 2])
    return float(diffs.mean()), float(lo), float(hi), float((boot > 0).mean())


m, lo, hi, p_pos = paired_bootstrap_ci(pq_df["MedCPT"].values, pq_df["BM25"].values)
print(f"\nMedCPT - BM25 (BioASQ): mean={m:+.4f} CI95=[{lo:+.4f}, {hi:+.4f}] P(MedCPT>BM25)={p_pos:.3f}")
boot_df = pd.DataFrame([{
    "contrast": "MedCPT - BM25",
    "dataset": "BioASQ-subset",
    "n_queries": len(qids),
    "mean_diff": round(m, 4),
    "ci_lo": round(lo, 4),
    "ci_hi": round(hi, 4),
    "p_a_gt_b": round(p_pos, 3),
}])
boot_df.to_csv(OUT / "tables" / "bioasq_paired_bootstrap.csv", index=False)
print(f"Saved bootstrap CI -> {OUT / 'tables' / 'bioasq_paired_bootstrap.csv'}")


---

## B.2 BGE cross-encoder reranking

Reranks the top-100 candidates from the strongest first-stage retriever for
each of the five datasets with `BAAI/bge-reranker-base`. A second reranker
pass (MedCPT cross-encoder) is applied on biomedical sets only.

Outputs:

- `notebooks/results/feedback2/tables/reranker_ndcg10.csv` (final-stage nDCG@10
   filled in for every row)
- `notebooks/results/feedback2/cache/reranked_perquery.parquet` (per-query
  nDCG@10 for the BGE-small + BGE-reranker@100 combo on NFCorpus, used by the router)


In [ ]:
# B.2.a Load the BGE cross-encoder reranker via sentence-transformers.
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
print(f"Loading {RERANKER_MODEL_NAME} ...")
reranker_model = CrossEncoder(RERANKER_MODEL_NAME, device=device, max_length=512)
print("BGE reranker ready (cross-encoder, scores (query, passage) pairs).")


In [ ]:
# B.2.b BGE cross-encoder reranker over precomputed top-100 candidates.

def cross_encoder_rerank(queries, corpus, candidate_ids_per_query, model, batch_size=32):
    """Score every (query, candidate) pair with a sentence-transformers
    CrossEncoder and return a dict-of-dicts in the same shape as the
    first-stage retrievers (so it slots into the existing harness).

    queries: dict[qid -> text]
    corpus: dict[doc_id -> {"title", "text"}]
    candidate_ids_per_query: dict[qid -> list[doc_id]] (top-100 from first-stage)
    Returns dict[qid -> dict[doc_id -> float]] reranked scores.
    """
    out = {}
    for qid in tqdm(queries, desc="BGE-reranker rerank"):
        cand_ids = candidate_ids_per_query.get(qid, [])
        if not cand_ids:
            out[qid] = {}
            continue
        cand_texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip()
                      for d in cand_ids]
        pairs = [(queries[qid], t) for t in cand_texts]
        scores = model.predict(pairs, batch_size=batch_size,
                                show_progress_bar=False,
                                convert_to_numpy=True)
        order = np.argsort(-np.asarray(scores, dtype=float))
        out[qid] = {cand_ids[i]: float(scores[i]) for i in order}
    return out


# First-stage candidate cache (we need to materialise top-100 per dataset).
# For NFCorpus we already have the per-query results in next_stage caches; for
# BioASQ we computed bioasq_results above; the other three datasets need a
# small first-stage pass which we run inline below.
def first_stage_top100(name, doc_ids, doc_texts, queries, retriever):
    if retriever == "BM25":
        return _bm25_run(doc_ids, doc_texts, queries, top_k=100)
    if retriever == "BGE-small":
        return _dense_run(doc_ids, doc_texts, queries, get_bge(), top_k=100)
    if retriever == "E5-small":
        return _dense_run(doc_ids, doc_texts, queries, get_e5(),
                          qpfx="query: ", dpfx="passage: ", top_k=100)
    raise ValueError(retriever)


RERANK_CACHE = CACHE_DIR / "reranked_results.pkl"
reranked_cache = pickle.load(open(RERANK_CACHE, "rb")) if RERANK_CACHE.exists() else {}


def _rerank_entry_ok(entry, queries):
    """A cached reranker entry is reusable only if it covers every query
    and at least one query produced a non-empty ranking. Empty {qid: {}}
    fan-outs from a partially-failed earlier run should be re-computed."""
    if not isinstance(entry, dict):
        return False
    if set(entry.keys()) != set(queries.keys()):
        return False
    return any(entry[qid] for qid in queries)


def _do_dataset(ds_name, corpus, queries, qrels, first_stage_results, first_stage_name):
    key = f"{ds_name}|{first_stage_name}|bge_reranker"
    if key in reranked_cache and _rerank_entry_ok(reranked_cache[key], queries):
        print(f"  cache hit: {key}")
        return reranked_cache[key]
    if key in reranked_cache:
        print(f"  cache hit was empty/incomplete; recomputing {key}")
    cand = {qid: list(first_stage_results[qid].keys()) for qid in queries}
    rer = cross_encoder_rerank(queries, corpus, cand, reranker_model)
    reranked_cache[key] = rer
    with open(RERANK_CACHE, "wb") as fh:
        pickle.dump(reranked_cache, fh)
    return rer


# We will fill these in dataset-by-dataset on Colab. Each call is cached.
print("Reranker harness ready. Run dataset-specific cells below.")


In [ ]:
# B.2.c Driver: run reranking over the 5 datasets.
# NFCorpus: reuse the cached per-query first-stage from final.ipynb if present;
# otherwise rebuild on the fly using the same code as final.ipynb.
def load_or_build_first_stage_nfcorpus(retriever_name):
    NF_DIR = DATASETS_DIR / "nfcorpus"
    if not NF_DIR.exists():
        beir_util.download_and_unzip(
            "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip",
            str(DATASETS_DIR))
    corpus, queries, qrels = GenericDataLoader(str(NF_DIR)).load(split="test")
    doc_ids, doc_texts = _prep_corpus(corpus)
    fs = first_stage_top100("NFCorpus", doc_ids, doc_texts, queries, retriever_name)
    return corpus, queries, qrels, fs


REranker_RESULTS = []  # rows for the CSV


def _agg_ndcg10(results, qrels):
    return evaluate_retriever(results, qrels)["aggregate"]["nDCG@10"]


# --- NFCorpus: BGE-small + BGE-reranker ---
print("\n[NFCorpus] BGE-small first-stage -> BGE-reranker rerank")
nf_corpus, nf_queries, nf_qrels, nf_fs_bge = load_or_build_first_stage_nfcorpus("BGE-small")
nf_rer_bge = _do_dataset("NFCorpus", nf_corpus, nf_queries, nf_qrels, nf_fs_bge, "BGE-small")
nf_first_bge = _agg_ndcg10(nf_fs_bge, nf_qrels)
nf_rer_bge_n = _agg_ndcg10(nf_rer_bge, nf_qrels)
print(f"  first={nf_first_bge:.4f}  reranked={nf_rer_bge_n:.4f}  delta={nf_rer_bge_n-nf_first_bge:+.4f}")
REranker_RESULTS.append(dict(Dataset="NFCorpus", FirstStage="BGE-small",
                              FirstStage_nDCG10=round(nf_first_bge, 4),
                              RerankedSystem="BGE-small+BGE-reranker@100",
                              Reranked_nDCG10=round(nf_rer_bge_n, 4),
                              Delta=round(nf_rer_bge_n - nf_first_bge, 4),
                              Status="done"))

# --- NFCorpus: BM25 + BGE-reranker ---
print("\n[NFCorpus] BM25 first-stage -> BGE-reranker rerank")
_, _, _, nf_fs_bm = load_or_build_first_stage_nfcorpus("BM25")
nf_rer_bm = _do_dataset("NFCorpus", nf_corpus, nf_queries, nf_qrels, nf_fs_bm, "BM25")
nf_first_bm = _agg_ndcg10(nf_fs_bm, nf_qrels)
nf_rer_bm_n = _agg_ndcg10(nf_rer_bm, nf_qrels)
print(f"  first={nf_first_bm:.4f}  reranked={nf_rer_bm_n:.4f}  delta={nf_rer_bm_n-nf_first_bm:+.4f}")
REranker_RESULTS.append(dict(Dataset="NFCorpus", FirstStage="BM25",
                              FirstStage_nDCG10=round(nf_first_bm, 4),
                              RerankedSystem="BM25+BGE-reranker@100",
                              Reranked_nDCG10=round(nf_rer_bm_n, 4),
                              Delta=round(nf_rer_bm_n - nf_first_bm, 4),
                              Status="done"))

# --- BioASQ-subset: BM25 + BGE-reranker ---
print("\n[BioASQ-subset] BM25 first-stage -> BGE-reranker rerank")
ba_rer_bm = _do_dataset("BioASQ-subset", bioasq_corpus, bioasq_queries, bioasq_qrels,
                         bioasq_results["BM25"], "BM25")
ba_first_bm = _agg_ndcg10(bioasq_results["BM25"], bioasq_qrels)
ba_rer_bm_n = _agg_ndcg10(ba_rer_bm, bioasq_qrels)
print(f"  first={ba_first_bm:.4f}  reranked={ba_rer_bm_n:.4f}  delta={ba_rer_bm_n-ba_first_bm:+.4f}")
REranker_RESULTS.append(dict(Dataset="BioASQ-subset", FirstStage="BM25",
                              FirstStage_nDCG10=round(ba_first_bm, 4),
                              RerankedSystem="BM25+BGE-reranker@100",
                              Reranked_nDCG10=round(ba_rer_bm_n, 4),
                              Delta=round(ba_rer_bm_n - ba_first_bm, 4),
                              Status="done"))

# --- TREC-COVID, SciFact, ArguAna with their best first-stage retriever ---
OTHER_DS = [
    ("TREC-COVID", "trec-covid", "E5-small"),
    ("SciFact",   "scifact",   "BGE-small"),
    ("ArguAna",   "arguana",   "BGE-small"),
]
for ds_name, beir_name, first_stage_name in OTHER_DS:
    print(f"\n[{ds_name}] {first_stage_name} first-stage -> BGE-reranker rerank")
    ds_dir = DATASETS_DIR / beir_name
    if not ds_dir.exists():
        beir_util.download_and_unzip(
            f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{beir_name}.zip",
            str(DATASETS_DIR))
    corpus, queries, qrels = GenericDataLoader(str(ds_dir)).load(split="test")
    doc_ids, doc_texts = _prep_corpus(corpus)
    fs = first_stage_top100(ds_name, doc_ids, doc_texts, queries, first_stage_name)
    rer = _do_dataset(ds_name, corpus, queries, qrels, fs, first_stage_name)
    fs_n = _agg_ndcg10(fs, qrels)
    rer_n = _agg_ndcg10(rer, qrels)
    print(f"  first={fs_n:.4f}  reranked={rer_n:.4f}  delta={rer_n-fs_n:+.4f}")
    REranker_RESULTS.append(dict(Dataset=ds_name, FirstStage=first_stage_name,
                                  FirstStage_nDCG10=round(fs_n, 4),
                                  RerankedSystem=f"{first_stage_name}+BGE-reranker@100",
                                  Reranked_nDCG10=round(rer_n, 4),
                                  Delta=round(rer_n - fs_n, 4),
                                  Status="done"))

# Save reranker_ndcg10.csv
rer_df = pd.DataFrame(REranker_RESULTS)
rer_df.to_csv(OUT / "tables" / "reranker_ndcg10.csv", index=False)
print(f"\nSaved -> {OUT / 'tables' / 'reranker_ndcg10.csv'}")
print(rer_df)


# Save NFCorpus per-query reranked nDCG@10 for the router (B.3).
def perquery_ndcg10(results, qrels):
    qids = sorted(set(results.keys()) & set(qrels.keys()))
    out = {}
    for qid in qids:
        ranking = [d for d, _ in sorted(results[qid].items(), key=lambda x: -x[1])]
        out[qid] = compute_ndcg(ranking, qrels[qid], 10)
    return out


nf_rer_bge_perq = perquery_ndcg10(nf_rer_bge, nf_qrels)
pd.DataFrame({"qid": list(nf_rer_bge_perq.keys()),
              "BGE_BGEReranker_at100": list(nf_rer_bge_perq.values())}
             ).to_parquet(CACHE_DIR / "reranked_perquery.parquet", index=False)
print(f"NFCorpus per-query reranked nDCG@10 -> {CACHE_DIR / 'reranked_perquery.parquet'}")


In [ ]:
# B.2.d MedCPT cross-encoder reranker on biomedical datasets only.
# Optional. Skip cleanly if the model fails to load on T4.
try:
    from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification as _AS
    medcpt_ce_tok = _AT.from_pretrained("ncbi/MedCPT-Cross-Encoder")
    medcpt_ce_mod = _AS.from_pretrained("ncbi/MedCPT-Cross-Encoder").to(device).eval()
    HAVE_MEDCPT_CE = True
except Exception as exc:
    print(f"MedCPT cross-encoder unavailable ({exc}); skipping this sub-section.")
    HAVE_MEDCPT_CE = False


def medcpt_ce_rerank(queries, corpus, candidate_ids_per_query, batch_size=16):
    out = {}
    for qid in tqdm(queries, desc="MedCPT-CE rerank"):
        cand_ids = candidate_ids_per_query.get(qid, [])
        if not cand_ids:
            out[qid] = {}
            continue
        cand_texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip()
                      for d in cand_ids]
        q = queries[qid]
        scores = []
        for i in range(0, len(cand_ids), batch_size):
            inp = medcpt_ce_tok([(q, t) for t in cand_texts[i:i + batch_size]],
                                 return_tensors="pt", truncation=True, padding=True,
                                 max_length=512).to(device)
            with torch.no_grad():
                lg = medcpt_ce_mod(**inp).logits.squeeze(-1).cpu().numpy()
            scores.extend(lg.tolist())
        order = np.argsort(-np.array(scores))
        out[qid] = {cand_ids[i]: float(scores[i]) for i in order}
    return out


if HAVE_MEDCPT_CE:
    medcpt_ce_rows = []

    # NFCorpus: BGE-small first-stage + MedCPT-CE
    rer = medcpt_ce_rerank(nf_queries, nf_corpus,
                           {qid: list(nf_fs_bge[qid].keys()) for qid in nf_queries})
    n_after = _agg_ndcg10(rer, nf_qrels)
    medcpt_ce_rows.append(dict(Dataset="NFCorpus", FirstStage="BGE-small",
                                 FirstStage_nDCG10=round(nf_first_bge, 4),
                                 RerankedSystem="BGE-small+MedCPT-CE@100",
                                 Reranked_nDCG10=round(n_after, 4),
                                 Delta=round(n_after - nf_first_bge, 4),
                                 Status="done"))

    # BioASQ-subset: BM25 + MedCPT-CE
    rer = medcpt_ce_rerank(bioasq_queries, bioasq_corpus,
                           {qid: list(bioasq_results["BM25"][qid].keys())
                            for qid in bioasq_queries})
    n_after = _agg_ndcg10(rer, bioasq_qrels)
    medcpt_ce_rows.append(dict(Dataset="BioASQ-subset", FirstStage="BM25",
                                 FirstStage_nDCG10=round(ba_first_bm, 4),
                                 RerankedSystem="BM25+MedCPT-CE@100",
                                 Reranked_nDCG10=round(n_after, 4),
                                 Delta=round(n_after - ba_first_bm, 4),
                                 Status="done"))

    rerank_df = pd.read_csv(OUT / "tables" / "reranker_ndcg10.csv")
    rerank_df = pd.concat([rerank_df, pd.DataFrame(medcpt_ce_rows)], ignore_index=True)
    rerank_df.to_csv(OUT / "tables" / "reranker_ndcg10.csv", index=False)
    print(rerank_df)


---

## B.3 Per-query router

Turns the conditional-heterogeneity story into an actual algorithm: predict
per query which retrieval strategy to use among
`{BM25, BGE-small, Hybrid(α=0.40), BGE-small + BGE-reranker@100}` from a small set
of cheap query features. Trained on NFCorpus train+dev queries, evaluated on
the held-out NFCorpus test split.

Outputs:

- `notebooks/results/feedback2/tables/router_test.csv` (final nDCG@10 for
  every router variant, plus oracle upper bound)
- `notebooks/results/feedback2/tables/router_train_test_gap.csv` (train vs test
  empirical risk for the chosen strategy — used by Section 4.10)
- `notebooks/results/feedback2/tables/router_feature_importance.csv`


In [ ]:
# B.3.a Build per-query features (gap, len, tech, querytype, medvocab, bm25top1).
NF_DIR = DATASETS_DIR / "nfcorpus"
if not NF_DIR.exists():
    beir_util.download_and_unzip(
        "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip",
        str(DATASETS_DIR))

# Load all three splits to mirror the existing train/dev/test partition.
splits = {}
for split in ["train", "dev", "test"]:
    corpus, queries, qrels = GenericDataLoader(str(NF_DIR)).load(split=split)
    splits[split] = {"corpus": corpus, "queries": queries, "qrels": qrels}

# All queries share the same corpus (NFCorpus), so prep once.
nf_doc_ids, nf_doc_texts = _prep_corpus(splits["test"]["corpus"])
nf_doc_id_to_idx = {d: i for i, d in enumerate(nf_doc_ids)}
nf_doc_text_by_id = dict(zip(nf_doc_ids, nf_doc_texts))

# Build a corpus-wide IDF table for the technicality feature.
import re
WORD_RE = re.compile(r"[A-Za-z][A-Za-z0-9\-]+")
def _tokens(s):
    return [t.lower() for t in WORD_RE.findall(s)]

import collections
df = collections.Counter()
for txt in nf_doc_texts:
    df.update(set(_tokens(txt)))
N = len(nf_doc_ids)
idf = {t: math.log((N + 1) / (df[t] + 1)) + 1.0 for t in df}

# Tiny medical-vocabulary heuristic: list of common UMLS-flavoured affixes /
# drug-name endings. This is a cheap proxy for "is this a clinical query?".
MED_AFFIX = ("mab", "tinib", "vir", "olol", "azepam", "azole", "statin", "cycline",
             "itis", "osis", "emia", "oma", "pathy", "cardio", "neuro", "hepato",
             "renal", "gastro", "pulmonary")

def _med_share(toks):
    if not toks:
        return 0.0
    return sum(any(t.endswith(x) or x in t for x in MED_AFFIX) for t in toks) / len(toks)

# Vocabulary gap = 1 - max overlap of query tokens with relevant docs.
def _gap(qtxt, rels):
    qset = set(_tokens(qtxt))
    if not qset:
        return 0.0
    best = 0.0
    for d in rels:
        if d not in nf_doc_text_by_id:
            continue
        dset = set(_tokens(nf_doc_text_by_id[d]))
        ov = len(qset & dset) / len(qset)
        best = max(best, ov)
    return 1.0 - best

# BM25 top-1 score (cheap "easy-query" signal).
import bm25s
_bm25_corpus_tok = bm25s.tokenize(nf_doc_texts, stopwords="en", show_progress=False)
_bm25_full = bm25s.BM25(k1=1.5, b=0.75)
_bm25_full.index(_bm25_corpus_tok)

def _bm25_top1_score(q):
    q_tok = bm25s.tokenize([q], stopwords="en", show_progress=False)
    # Force single-threaded retrieval (see _bm25_run docstring): the bm25s
    # 0.2.x default multiprocessing pool occasionally returns zero items on
    # Colab kernels and triggers `not enough values to unpack`.
    try:
        _, sc = _bm25_full.retrieve(q_tok, k=1, n_threads=1, show_progress=False)
    except TypeError:
        _, sc = _bm25_full.retrieve(q_tok, k=1)
    if len(sc) == 0 or len(sc[0]) == 0:
        return 0.0
    return float(sc[0][0])

QUESTION_RE = re.compile(r"^\s*(what|who|how|why|when|where|is|are|does|do|can|should)\b", re.I)


feature_rows = []
for split, d in splits.items():
    for qid, qtxt in d["queries"].items():
        toks = _tokens(qtxt)
        rels = d["qrels"].get(qid, {})
        feature_rows.append({
            "split": split, "qid": qid,
            "gap": _gap(qtxt, rels),
            "len": len(toks),
            "tech": float(np.mean([idf.get(t, 1.0) for t in toks])) if toks else 0.0,
            "is_question": int(bool(QUESTION_RE.match(qtxt))),
            "med_share": _med_share(toks),
            "bm25_top1": _bm25_top1_score(qtxt),
        })
features_df = pd.DataFrame(feature_rows)
print("features:", features_df.shape)
features_df.head()


In [ ]:
# B.3.b Per-query nDCG@10 for each strategy on every NFCorpus query.
# Strategies: BM25, BGE-small, Hybrid(alpha=0.40), BGE-small+BGE-reranker@100.

ALPHA_HYBRID = 0.40

def _per_q_ndcg10(results, qrels):
    out = {}
    for qid, doc_scores in results.items():
        if qid not in qrels:
            continue
        ranking = [d for d, _ in sorted(doc_scores.items(), key=lambda x: -x[1])]
        out[qid] = compute_ndcg(ranking, qrels[qid], 10)
    return out


# (1) Run BM25 + BGE-small over each split (cached)
ROUTER_CACHE = CACHE_DIR / "router_strategy_perq.parquet"
_router_cache_ok = False
if ROUTER_CACHE.exists():
    try:
        _candidate = pd.read_parquet(ROUTER_CACHE)
        _need_cols = {"split", "qid", "BM25", "BGE-small", "Hybrid", "BGE+BGE-reranker"}
        _need_splits = {"train", "dev", "test"}
        if (_need_cols.issubset(_candidate.columns) and
                _need_splits.issubset(set(_candidate["split"].unique()))):
            perq_strategies = _candidate
            _router_cache_ok = True
            print("Loaded per-query strategy nDCG@10 cache.")
        else:
            print("Router cache missing splits/columns; rebuilding.")
    except Exception as exc:
        print(f"Router cache unreadable ({exc}); rebuilding.")

if not _router_cache_ok:
    bge = get_bge()
    rows = []
    for split, d in splits.items():
        bm25_res = _bm25_run(nf_doc_ids, nf_doc_texts, d["queries"], top_k=100)
        bge_res  = _dense_run(nf_doc_ids, nf_doc_texts, d["queries"], bge, top_k=100)
        # Hybrid: per-query min-max normalise, then weighted sum.
        def _mm(a):
            lo, hi = a.min(), a.max()
            return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

        hyb_res = {}
        for qid in d["queries"]:
            sb = bm25_res.get(qid, {})
            sd = bge_res.get(qid, {})
            # Deterministic ordering: pin the candidate union to a list so
            # `sb_arr`, `sd_arr` and the score-to-doc-id zip below all share
            # the same index sequence regardless of set hash ordering.
            cand = sorted(set(sb) | set(sd))
            if not cand:
                hyb_res[qid] = {}
                continue
            sb_arr = np.array([sb.get(c, 0.0) for c in cand])
            sd_arr = np.array([sd.get(c, 0.0) for c in cand])
            hyb = ALPHA_HYBRID * _mm(sb_arr) + (1.0 - ALPHA_HYBRID) * _mm(sd_arr)
            hyb_res[qid] = {c: float(s) for c, s in zip(cand, hyb)}

        # BGE cross-encoder reranker over BGE-small top-100 (per split).
        cand = {qid: list(bge_res[qid].keys()) for qid in d["queries"]}
        rer = cross_encoder_rerank(d["queries"], d["corpus"], cand, reranker_model)

        bm25_q = _per_q_ndcg10(bm25_res, d["qrels"])
        bge_q  = _per_q_ndcg10(bge_res,  d["qrels"])
        hyb_q  = _per_q_ndcg10(hyb_res,  d["qrels"])
        rer_q  = _per_q_ndcg10(rer,      d["qrels"])
        for qid in d["queries"]:
            rows.append({"split": split, "qid": qid,
                          "BM25": bm25_q.get(qid, 0.0),
                          "BGE-small": bge_q.get(qid, 0.0),
                          "Hybrid": hyb_q.get(qid, 0.0),
                          "BGE+BGE-reranker": rer_q.get(qid, 0.0)})
    perq_strategies = pd.DataFrame(rows)
    perq_strategies.to_parquet(ROUTER_CACHE, index=False)

print(perq_strategies.groupby("split")[["BM25", "BGE-small", "Hybrid", "BGE+BGE-reranker"]].mean().round(4))


In [ ]:
# B.3.c Train logistic + LightGBM router; report Always-baselines and oracle.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb

STRATS = ["BM25", "BGE-small", "Hybrid", "BGE+BGE-reranker"]

# Join features and labels
df_all = features_df.merge(perq_strategies, on=["split", "qid"], how="inner")
labels = df_all[STRATS].values.argmax(axis=1)
df_all["label"] = labels
print("strategy distribution (label = argmax over 4 strategies):")
print(pd.Series(labels).value_counts().rename(lambda i: STRATS[i]))

FEATURES = ["gap", "len", "tech", "is_question", "med_share", "bm25_top1"]

train_mask = df_all["split"].isin(["train", "dev"])
test_mask  = df_all["split"] == "test"

Xtr = df_all.loc[train_mask, FEATURES].values
ytr = df_all.loc[train_mask, "label"].values
Xte = df_all.loc[test_mask,  FEATURES].values
yte = df_all.loc[test_mask,  "label"].values

# Logistic regression. The default `lbfgs` solver in modern scikit-learn
# (>= 1.5) is multinomial out-of-the-box; passing `multi_class="multinomial"`
# now triggers a FutureWarning and will be removed.
log_pipe = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=2000))
log_pipe.fit(Xtr, ytr)

# LightGBM
gbm = lgb.LGBMClassifier(num_leaves=31, learning_rate=0.05, n_estimators=300,
                          objective="multiclass", num_class=4, random_state=42)
gbm.fit(Xtr, ytr)


def _route(model, X, df_split):
    """Return mean nDCG@10 if we always pick model.predict(X) per query."""
    preds = model.predict(X)
    chosen = df_split[STRATS].values[np.arange(len(df_split)), preds]
    return float(np.mean(chosen)), preds


# --- Test-set evaluation ---
test_df = df_all.loc[test_mask].reset_index(drop=True)

log_test, log_preds_te = _route(log_pipe, Xte, test_df)
gbm_test, gbm_preds_te = _route(gbm,      Xte, test_df)
oracle = float(np.mean(test_df[STRATS].values.max(axis=1)))
always_bge = float(np.mean(test_df["BGE-small"].values))
always_hyb = float(np.mean(test_df["Hybrid"].values))
always_rer = float(np.mean(test_df["BGE+BGE-reranker"].values))
always_bm  = float(np.mean(test_df["BM25"].values))

router_rows = [
    {"Method": "Always_BM25", "nDCG@10": round(always_bm,  4), "Status": "baseline"},
    {"Method": "Always_BGE_small", "nDCG@10": round(always_bge, 4), "Status": "baseline"},
    {"Method": "Static_Hybrid_alpha_0.40", "nDCG@10": round(always_hyb, 4), "Status": "baseline"},
    {"Method": "Always_BGE_small+BGE_reranker", "nDCG@10": round(always_rer, 4), "Status": "baseline"},
    {"Method": "Logistic_router",  "nDCG@10": round(log_test, 4), "Status": "learned"},
    {"Method": "LightGBM_router",  "nDCG@10": round(gbm_test, 4), "Status": "learned"},
    {"Method": "Oracle_router (upper bound)", "nDCG@10": round(oracle, 4), "Status": "oracle"},
]
router_df = pd.DataFrame(router_rows)
router_df.to_csv(OUT / "tables" / "router_test.csv", index=False)
print(router_df)


# Feature importance (LightGBM gain-based)
fi = pd.DataFrame({"feature": FEATURES,
                    "gain": gbm.booster_.feature_importance(importance_type="gain")})
fi.sort_values("gain", ascending=False, inplace=True)
fi.to_csv(OUT / "tables" / "router_feature_importance.csv", index=False)
print("\nFeature importance (gain):")
print(fi)


In [ ]:
# B.3.d Router train/test generalisation gap (Section 4.10 anchor).
train_df = df_all.loc[train_mask].reset_index(drop=True)
log_train, _ = _route(log_pipe, Xtr, train_df)
gbm_train, _ = _route(gbm,      Xtr, train_df)

def _risk(mean_ndcg10):
    return 1.0 - mean_ndcg10

gap_rows = [
    {"Router": "Logistic_router",
     "R_train": round(_risk(log_train), 4),
     "R_test":  round(_risk(log_test), 4),
     "Gap":     round(_risk(log_test) - _risk(log_train), 4)},
    {"Router": "LightGBM_router",
     "R_train": round(_risk(gbm_train), 4),
     "R_test":  round(_risk(gbm_test), 4),
     "Gap":     round(_risk(gbm_test) - _risk(gbm_train), 4)},
]
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT / "tables" / "router_train_test_gap.csv", index=False)
print(gap_df)


---

## B.4 MIRAGE/MedRAG downstream

Two medical QA datasets with gold answers (PubMedQA-labeled and BioASQ Y/N),
four retrieval settings (`None`, `BM25`, `BGE-small`, `BGE-small+BGE-reranker@100`),
two generators (`Llama-3.2-3B-Instruct` 4-bit and `flan-t5-base`).

Outputs:

- `notebooks/results/feedback2/tables/mirage_accuracy.csv` (filled)
- `notebooks/results/feedback2/cache/mirage_predictions.parquet` (raw answers)


In [ ]:
# B.4.a Load PubMedQA-labeled and BioASQ-Y/N.
from datasets import load_dataset

pq = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(f"PubMedQA-labeled: {len(pq)}")
# Each row has question, context (list of dicts), final_decision in {yes,no,maybe}
pubmedqa = []
for r in pq:
    pubmedqa.append({
        "qid": str(r["pubid"]),
        "question": r["question"],
        "answer": r["final_decision"].lower().strip(),
        "context_passages": r["context"]["contexts"],
    })


# BioASQ-Y/N: filter Yes/No questions from BioASQ training data on HuggingFace.
try:
    ba = load_dataset("kroshan/BioASQ", split="train")
    bioasq_yn = []
    rng = np.random.default_rng(42)
    for r in ba:
        if r.get("type", "").lower() == "yesno" and r.get("answer", "").lower() in ("yes", "no"):
            bioasq_yn.append({
                "qid": r.get("id", str(len(bioasq_yn))),
                "question": r["body"],
                "answer": r["answer"].lower(),
            })
    if len(bioasq_yn) > 500:
        idx = rng.choice(len(bioasq_yn), size=500, replace=False)
        bioasq_yn = [bioasq_yn[i] for i in sorted(idx.tolist())]
    print(f"BioASQ-Y/N: {len(bioasq_yn)}")
except Exception as exc:
    print(f"BioASQ-Y/N load failed ({exc}); will fall back to BioASQ-subset queries.")
    # Fallback: use the BioASQ-subset queries as a question source (no Y/N gold, skip later).
    bioasq_yn = []


In [ ]:
# B.4.b Build top-5 evidence passages per question for each retrieval setting.
# Reuse PubMedQA's own context_passages for the closed-book and BM25/BGE
# settings; that's the standard MIRAGE/MedRAG protocol.

def build_pubmedqa_index():
    """Single concatenated index of all PubMedQA contexts."""
    docs = {}
    for r in pubmedqa:
        for ci, passage in enumerate(r["context_passages"]):
            docs[f"{r['qid']}_p{ci}"] = {"title": "", "text": passage,
                                          "owner_qid": r["qid"]}
    return docs


pubmedqa_corpus = build_pubmedqa_index()
pq_doc_ids = list(pubmedqa_corpus.keys())
pq_doc_texts = [pubmedqa_corpus[d]["text"] for d in pq_doc_ids]
print(f"PubMedQA pseudo-corpus: {len(pq_doc_ids):,} passages")


def topk_for_questions(rows, retriever_name, k=5):
    queries = {r["qid"]: r["question"] for r in rows}
    if retriever_name == "BM25":
        res = _bm25_run(pq_doc_ids, pq_doc_texts, queries, top_k=k)
    elif retriever_name == "BGE-small":
        res = _dense_run(pq_doc_ids, pq_doc_texts, queries, get_bge(), top_k=k)
    elif retriever_name == "BGE-small+BGE-reranker":
        first = _dense_run(pq_doc_ids, pq_doc_texts, queries, get_bge(), top_k=100)
        cand = {qid: list(first[qid].keys()) for qid in queries}
        rer = cross_encoder_rerank(queries, pubmedqa_corpus, cand, reranker_model)
        res = {qid: dict(list(rer[qid].items())[:k]) for qid in queries}
    else:
        raise ValueError(retriever_name)
    return {qid: [pubmedqa_corpus[d]["text"] for d in res[qid]] for qid in queries}


In [ ]:
# B.4.c Load generators and run both Llama-3.2-3B-Instruct (4-bit) and flan-t5-base.
import re as _re

def _ensure(pkgs):
    try:
        for p in pkgs:
            importlib.import_module(p.split(">=")[0].split("==")[0])
    except ImportError:
        _pip(pkgs)


_ensure(["bitsandbytes>=0.43.0", "accelerate>=0.32.0", "sentencepiece"])

from transformers import (AutoTokenizer, AutoModelForCausalLM,
                           AutoModelForSeq2SeqLM, BitsAndBytesConfig)


def load_llama():
    name = "meta-llama/Llama-3.2-3B-Instruct"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(name)
    mod = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb,
                                                 device_map="auto")
    return tok, mod


def load_flan():
    name = "google/flan-t5-base"
    tok = AutoTokenizer.from_pretrained(name)
    mod = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    return tok, mod


PROMPT_TMPL = (
    "You are a biomedical question-answering assistant.\n"
    "Answer with one of: yes, no, maybe.\n"
    "Question: {q}\n"
    "Evidence:\n{ctx}\n"
    "Final answer:"
)


def _format_prompt(q, passages):
    ctx = "\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages or []))
    if not ctx:
        ctx = "(no retrieved evidence)"
    return PROMPT_TMPL.format(q=q, ctx=ctx)


def _normalize_answer(text):
    t = text.strip().lower()
    for label in ("yes", "no", "maybe"):
        if _re.search(rf"\b{label}\b", t):
            return label
    return "unknown"


def _gen_llama(tok, mod, prompt, max_new=8):
    msgs = [{"role": "user", "content": prompt}]
    inp = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(mod.device)
    with torch.no_grad():
        out = mod.generate(inp, max_new_tokens=max_new, do_sample=False)
    txt = tok.decode(out[0, inp.shape[-1]:], skip_special_tokens=True)
    return _normalize_answer(txt)


def _gen_flan(tok, mod, prompt, max_new=4):
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(mod.device)
    with torch.no_grad():
        out = mod.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return _normalize_answer(tok.decode(out[0], skip_special_tokens=True))


# Configurations to evaluate
ROWS = []
PRED_ROWS = []
RETRIEVERS = ["None (closed-book)", "BM25", "BGE-small", "BGE-small+BGE-reranker"]
TASKS = [("PubMedQA-labeled", pubmedqa)]
if bioasq_yn:
    TASKS.append(("BioASQ-Y/N", bioasq_yn))


print("Loading flan-t5-base ...")
ft_tok, ft_mod = load_flan()
print("Loading Llama-3.2-3B-Instruct (4-bit) ...")
# Llama-3.2-3B-Instruct is a gated model: the user must `huggingface-cli
# login` and accept the licence on the model page. If that step is
# missing the load fails with a 401/403 -- skip Llama and report only
# flan-t5 results in that case.
try:
    ll_tok, ll_mod = load_llama()
    HAVE_LLAMA = True
except Exception as exc:
    print(f"  Llama-3.2 unavailable ({type(exc).__name__}: {exc}).")
    print(f"  Falling back to flan-t5-base only. To enable Llama, run "
          f"`huggingface-cli login` and accept the licence at "
          f"https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct .")
    ll_tok = ll_mod = None
    HAVE_LLAMA = False


for task_name, task_rows in TASKS:
    # Pre-compute retrieval contexts for the three real retrievers.
    contexts = {"None (closed-book)": {r["qid"]: [] for r in task_rows}}
    if task_name == "PubMedQA-labeled":
        for retr in ["BM25", "BGE-small", "BGE-small+BGE-reranker"]:
            print(f"  retrieval[{retr}] ...", flush=True)
            contexts[retr] = topk_for_questions(task_rows, retr, k=5)
    else:
        # BioASQ-Y/N: reuse the BioASQ-subset corpus we built in B.1.
        for retr in ["BM25", "BGE-small", "BGE-small+BGE-reranker"]:
            print(f"  retrieval[{retr}] (BioASQ corpus) ...", flush=True)
            queries = {r["qid"]: r["question"] for r in task_rows}
            if retr == "BM25":
                top = _bm25_run(doc_ids_b, doc_texts_b, queries, top_k=5)
            elif retr == "BGE-small":
                top = _dense_run(doc_ids_b, doc_texts_b, queries, get_bge(), top_k=5)
            else:
                first = _dense_run(doc_ids_b, doc_texts_b, queries, get_bge(), top_k=100)
                cand = {qid: list(first[qid].keys()) for qid in queries}
                rer = cross_encoder_rerank(queries, bioasq_corpus, cand, reranker_model)
                top = {qid: dict(list(rer[qid].items())[:5]) for qid in queries}
            contexts[retr] = {qid: [(bioasq_corpus[d].get("title", "") + " " + bioasq_corpus[d].get("text", "")).strip()
                                     for d in top[qid]] for qid in queries}

    _gens = [("flan-t5-base", ft_tok, ft_mod, _gen_flan)]
    if HAVE_LLAMA:
        _gens.append(("Llama-3.2-3B-Instruct", ll_tok, ll_mod, _gen_llama))

    for retr in RETRIEVERS:
        for gen_name, tok, mod, gen_fn in _gens:
            correct = 0
            total = 0
            for r in tqdm(task_rows, desc=f"{task_name} | {retr} | {gen_name}", leave=False):
                ctx = contexts[retr].get(r["qid"], [])
                ans = gen_fn(tok, mod, _format_prompt(r["question"], ctx))
                gold = r["answer"].lower()
                if task_name == "BioASQ-Y/N" and ans == "maybe":
                    ans = "yes"  # cap to {yes,no}
                if ans == gold:
                    correct += 1
                total += 1
                PRED_ROWS.append({"task": task_name, "retriever": retr,
                                   "generator": gen_name, "qid": r["qid"],
                                   "gold": gold, "prediction": ans})
            acc = correct / total if total else 0.0
            ROWS.append({"Task": task_name, "Retriever": retr,
                          "Generator": gen_name,
                          "Accuracy": round(acc, 4),
                          "Status": "done"})
            print(f"  {task_name} | {retr} | {gen_name}: acc = {acc:.4f}")


mirage_df = pd.DataFrame(ROWS)
mirage_df.to_csv(OUT / "tables" / "mirage_accuracy.csv", index=False)
print(mirage_df)

pred_df = pd.DataFrame(PRED_ROWS)
pred_df.to_parquet(CACHE_DIR / "mirage_predictions.parquet", index=False)
print(f"Predictions cached -> {CACHE_DIR / 'mirage_predictions.parquet'}")


---

## Phase-C handoff

After all sections above complete on Colab T4, sync the produced
`notebooks/results/feedback2/` directory back to the local repo. The
artifacts referenced in `main.tex` are:

- `tables/multi_dataset_ndcg10_v2.csv` (BioASQ row filled)
- `tables/reranker_ndcg10.csv` (BGE-reranker + MedCPT-CE rows filled)
- `tables/router_test.csv` (logistic / LightGBM / oracle rows filled)
- `tables/router_train_test_gap.csv`, `tables/router_feature_importance.csv`
- `tables/mirage_accuracy.csv` (8 rows per dataset filled)
- `tables/bioasq_paired_bootstrap.csv`
- `cache/bioasq_perquery.parquet`, `cache/reranked_perquery.parquet`,
  `cache/mirage_predictions.parquet`

Phase D of the project plan then backfills `main.tex` Tables 4, 5, 6, 8 with
these numbers.
